In [1]:
import yfinance as yf
import pandas as pd
import time
import requests
from requests.exceptions import HTTPError

# Configure custom session with browser-like headers
session = requests.Session()
session.headers.update({
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
})

# Stock parameters
ticker_symbol = "M&M.BO"
start_date = "2007-01-01"
end_date = "2025-12-31"

def fetch_stock_data(max_retries=5, initial_delay=10):
    current_delay = initial_delay
    for attempt in range(max_retries):
        try:
            stock = yf.Ticker(ticker_symbol, session=session)
            data = stock.history(
                start=start_date,
                end=end_date,
                interval="1d",
                actions=False
            )
            
            if data.empty:
                raise ValueError("No data returned - check dates/ticker")
                
            processed_data = data[['Open', 'Close', 'Volume']].reset_index()
            processed_data['Date'] = pd.to_datetime(processed_data['Date']).dt.date
            return processed_data
            
        except Exception as e:
            print(f"Attempt {attempt+1} failed: {str(e)}")
            if attempt < max_retries - 1:
                print(f"Waiting {current_delay} seconds...")
                time.sleep(current_delay)
                current_delay *= 2  # Exponential backoff
                
    print("All retries failed. Try again later.")
    return None

# Main execution
if __name__ == "__main__":
    print(f"Fetching {ticker_symbol} data from {start_date} to {end_date}")
    stock_data = fetch_stock_data()
    
    if stock_data is not None:
        print("\nFirst 5 rows:")
        print(stock_data.head())
        #stock_data.to_csv("M&M.BO_Stock_Data.csv", index=False)
        #print("\nSaved to M&M.BO.NS_Stock_Data.csv")
        print(f"\nDate Range: {stock_data['Date'].min()} to {stock_data['Date'].max()}")
    else:
        print("\nFailed to fetch data. Possible solutions:")
        print("- Use a VPN/proxy")
        print("- Verify ticker at https://finance.yahoo.com/quote/BA.L")
        print("- Wait 1 hour and try again")

Fetching M&M.BO data from 2007-01-01 to 2025-12-31

First 5 rows:
         Date        Open       Close   Volume
0  2007-01-02  140.902173  147.715530   929016
1  2007-01-03  148.318098  144.293427  1682768
2  2007-01-04  144.610093  143.026489   708168
3  2007-01-05  143.528631  138.507446   979996
4  2007-01-08  140.562266  138.901413   645708

Date Range: 2007-01-02 to 2025-02-27


In [2]:
import yfinance as yf
import pandas as pd
import time
import requests
import random
from requests.exceptions import HTTPError

# Configure custom session with browser-like headers
session = requests.Session()
session.headers.update({
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
})

# Ticker list (FTSE 100 companies)
TICKERS = [
            "M&M.BO", "MARUTI.BO", "TATAMOTORS.BO", "AXISBANK.BO", "HDFCBANK.BO", "ICICIBANK.BO", "INDUSINDBK.BO", "KOTAKBANK.BO",
    "SBIN.BO", "ULTRACEMCO.BO", "ITC.BO", "RELIANCE.BO", "TITAN.BO", "ZOMATO.BO", "LT.BO", "BAJAJFINSV.BO",
    "BAJFINANCE.BO", "HINDUNILVR.BO", "NESTLEIND.BO", "HCLTECH.BO", "INFY.BO", "TCS.BO", "TECHM.BO", "ASIANPAINT.BO",
    "SUNPHARMA.BO", "ADANIPORTS.BO", "NTPC.BO", "POWERGRID.BO", "TATASTEEL.BO", "BHARTIARTL.BO"
]

# Date parameters
START_DATE = "2007-01-01"
END_DATE = "2025-12-31"

def fetch_stock_data(ticker, max_retries=3, initial_delay=5):
    current_delay = initial_delay
    for attempt in range(max_retries + 1):
        try:
            stock = yf.Ticker(ticker, session=session)
            data = stock.history(
                start=START_DATE,
                end=END_DATE,
                interval="1d",
                actions=False
            )
            
            if data.empty:
                print(f"No data for {ticker}")
                return None
                
            processed_data = data[['Open', 'Close', 'Volume']].reset_index()
            processed_data['Date'] = pd.to_datetime(processed_data['Date']).dt.date
            processed_data['Ticker'] = ticker  # Add Ticker column
            return processed_data
            
        except Exception as e:
            if attempt < max_retries:
                sleep_time = current_delay + random.uniform(0, 3)
                print(f"Retry {attempt+1} for {ticker} in {sleep_time:.1f}s: {str(e)}")
                time.sleep(sleep_time)
                current_delay *= 2
            else:
                print(f"Failed {ticker} after {max_retries} retries")
                return None

def main():
    all_data = []
    total_tickers = len(TICKERS)
    success_count = 0
    
    for idx, ticker in enumerate(TICKERS, 1):
        print(f"\nProcessing {ticker} ({idx}/{total_tickers})")
        
        # Random delay to avoid detection
        time.sleep(random.uniform(0.5, 1.5))
        
        data = fetch_stock_data(ticker)
        
        if data is not None:
            all_data.append(data)
            success_count += 1
            
    if all_data:
        # Merge all DataFrames
        combined_df = pd.concat(all_data, ignore_index=True)
        
        # Save merged data
        combined_df.to_csv("BSESN_companies_stock.csv", index=False)
        print(f"\nSuccessfully saved {len(combined_df)} rows from {success_count} companies")
        print("Columns in final dataset:", combined_df.columns.tolist())
    else:
        print("\nNo data was collected")
        
    print(f"\nSuccess rate: {success_count}/{total_tickers} ({success_count/total_tickers:.1%})")

if __name__ == "__main__":
    start_time = time.time()
    main()
    print(f"\nTotal execution time: {(time.time() - start_time)/60:.2f} minutes")


Processing M&M.BO (1/30)

Processing MARUTI.BO (2/30)

Processing TATAMOTORS.BO (3/30)

Processing AXISBANK.BO (4/30)

Processing HDFCBANK.BO (5/30)

Processing ICICIBANK.BO (6/30)

Processing INDUSINDBK.BO (7/30)

Processing KOTAKBANK.BO (8/30)

Processing SBIN.BO (9/30)

Processing ULTRACEMCO.BO (10/30)

Processing ITC.BO (11/30)

Processing RELIANCE.BO (12/30)

Processing TITAN.BO (13/30)

Processing ZOMATO.BO (14/30)

Processing LT.BO (15/30)

Processing BAJAJFINSV.BO (16/30)

Processing BAJFINANCE.BO (17/30)

Processing HINDUNILVR.BO (18/30)

Processing NESTLEIND.BO (19/30)

Processing HCLTECH.BO (20/30)

Processing INFY.BO (21/30)

Processing TCS.BO (22/30)

Processing TECHM.BO (23/30)

Processing ASIANPAINT.BO (24/30)

Processing SUNPHARMA.BO (25/30)

Processing ADANIPORTS.BO (26/30)

Processing NTPC.BO (27/30)

Processing POWERGRID.BO (28/30)

Processing TATASTEEL.BO (29/30)

Processing BHARTIARTL.BO (30/30)

Successfully saved 129234 rows from 30 companies
Columns in final da